# Data Preprocessing


In [ ]:
import numpy as np 
import pandas as pd 

df=pd.read_csv("Renewable.csv");
print(df.shape);
print(df.head());

print(df.duplicated().sum());
print(df.isnull().sum());

df["Time"] = pd.to_datetime(df["Time"],format="%d-%m-%Y %H:%M")
df = df.sort_values("Time").reset_index(drop=True)
print(df);

print(df["Time"].head())
print(df["Time"].dtype)

# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check duplicate timestamps
print("Duplicate timestamps:", df["Time"].duplicated().sum())


In [ ]:
columns_to_drop = [
    "isSun",
    "sunlightTime",
    "dayLength",
    "SunlightTime/daylength",
    "hour",
    "month",
    "snow_1h"
]

df.drop(columns=columns_to_drop, inplace=True, errors="ignore")

print(df.columns.tolist())

# Convert interval energy into average power in Watts
df["Power_W"] = df["Energy"] * 4

# Display the result
print(df[["Energy", "Power_W"]].head())

df["hour"] = df["Time"].dt.hour

df["day_of_year"] = df["Time"].dt.dayofyear

df["day_of_week"] = df["Time"].dt.dayofweek

df["month"] = df["Time"].dt.month

print(df[["Time", "hour", "month", "day_of_year","day_of_week"]].head(10))

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)

df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["day_of_year_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["day_of_year_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

df.drop(columns=["Energy"], inplace=True)

print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# Exploratory Data Analysis (Analyzing Data)


In [ ]:
df.info()

# Display statistical summary
df.describe()

print("Average Power:", df["Power_W"].mean())

print("Maximum Power:", df["Power_W"].max())

print("Minimum Power:", df["Power_W"].min())

print("Standard Deviation:", df["Power_W"].std())

# Distribution of Power_W
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

sns.histplot(df["Power_W"], bins=50, kde=True)

plt.title("Distribution of Solar Power Generation")
plt.xlabel("Power (W)")
plt.ylabel("Frequency")

plt.show()

# Average Power_W for each hour
plt.figure(figsize=(12, 5))

sns.lineplot(
    data=df,
    x="hour",
    y="Power_W",
    estimator="mean",
    errorbar=None
)

plt.title("Average Solar Power Generation by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Power (W)")

plt.xticks(range(0, 24))
plt.grid(True)

plt.show()


In [ ]:
# Which environmental features have a strong relationship with solar power generation

# | Correlation | Interpretation |
# | +1.00 | Perfect positive relationship |
# | +0.70 | Strong positive relationship |
# | +0.00 | No linear relationship |
# | -0.70 | Strong negative relationship |
# | -1.00 | Perfect negative relationship |

# Select numerical columns
numeric_df = df.select_dtypes(include="number")

# Calculate correlation
correlation_matrix = numeric_df.corr()

# Visualize correlation
plt.figure(figsize=(12, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of Solar Power Generation")

plt.show()

# visualizing the strongest relationship (GHI)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="GHI",
    y="Power_W",
    alpha=0.3
)

plt.title("Relationship Between GHI and Solar Power Generation")
plt.xlabel("Global Horizontal Irradiance (GHI)")
plt.ylabel("Average Power (W)")

plt.grid(True)

plt.show()

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="temp",
    y="Power_W",
    alpha=0.3
)

plt.title("Relationship Between Temperature and Solar Power Generation")
plt.xlabel("Temperature")
plt.ylabel("Average Power (W)")

plt.grid(True)

plt.show()

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="clouds_all",
    y="Power_W",
    alpha=0.3
)

plt.title("Relationship Between Cloud Coverage and Solar Power Generation")
plt.xlabel("Cloud Coverage")
plt.ylabel("Average Power (W)")

plt.grid(True)

plt.show()

# Average power by month

plt.figure(figsize=(12, 5))

sns.barplot(
    data=df,
    x="month",
    y="Power_W",
    estimator="mean",
    errorbar=None
)

plt.title("Average Solar Power Generation by Month")
plt.xlabel("Month")
plt.ylabel("Average Power (W)")

plt.grid(axis="y")

plt.show()

# Set Time as the index
daily_power = df.set_index("Time")["Power_W"].resample("D").mean()

# Plot daily average power
plt.figure(figsize=(15, 6))

daily_power.plot()

plt.title("Daily Average Solar Power Generation Over Time")
plt.xlabel("Date")
plt.ylabel("Average Power (W)")

plt.grid(True)

plt.show()


In [ ]:
print(df["weather_type"].unique())

print("\nNumber of weather categories:")
print(df["weather_type"].nunique())

plt.figure(figsize=(10, 6))

sns.barplot(
    data=df,
    x="weather_type",
    y="Power_W",
    estimator="mean",
    errorbar=None
)

plt.title("Average Solar Power Generation by Weather Type")
plt.xlabel("Weather Type")
plt.ylabel("Average Power (W)")

plt.grid(axis="y")

plt.show()

df.groupby("weather_type")["Power_W"].agg(
    ["count", "mean", "median", "min", "max"]
)

# EDA SUMMARY

print("Total observations:", len(df))

print("\nPower Statistics:")
print(df["Power_W"].describe())

print("\nZero Power Observations:")
print((df["Power_W"] == 0).sum())

print("\nNegative Power Observations:")
print((df["Power_W"] < 0).sum())


# Feature Engineering & Model Preparation


In [ ]:
df.columns

df.head()

# select input features

# Select input features
X = df[
    [
        "GHI",
        "temp",
        "pressure",
        "humidity",
        "wind_speed",
        "rain_1h",
        "clouds_all",
        "weather_type",
        "hour_sin",
        "hour_cos",
        "day_of_year_sin",
        "day_of_year_cos",
        "day_of_week"
    ]
]

# Select target variable
y = df["Power_W"]

# Preserve timestamps for chronological splitting and analysis
timestamps = df["Time"]

# Check shapes
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Timestamps shape:", timestamps.shape)

# Preview
X.head()

print(X["weather_type"].value_counts().sort_index())

# One-Hot Encode weather_type
X_encoded = pd.get_dummies(
    X,
    columns=["weather_type"],
    dtype=int
)

# Check the result
print("Original X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)

print("\nNew columns:")
print(X_encoded.columns.tolist())

X_encoded.head()


In [ ]:
from sklearn.preprocessing import StandardScaler

# Calculate split index
split_index = int(len(X_encoded) * 0.80)

# Split input features
X_train = X_encoded.iloc[:split_index]
X_test = X_encoded.iloc[split_index:]

# Split target variable
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Preserve timestamps
time_train = timestamps.iloc[:split_index]
time_test = timestamps.iloc[split_index:]

# Display dataset shapes
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

# Display date ranges
print("\nTraining Period:")
print(time_train.min(), "to", time_train.max())

print("\nTesting Period:")
print(time_test.min(), "to", time_test.max())

# Feature scaling
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data using training statistics
X_test_scaled = scaler.transform(X_test)

# Check shapes
print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

# Check scaled feature values
print("\nFirst 5 scaled observations:")
print(X_train_scaled[:5])


# Model Training and Evaluation


In [ ]:
from sklearn.linear_model import LinearRegression

# Create Linear Regression model
lr_model = LinearRegression()

# Train the model
lr_model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully!")

y_pred_lr = lr_model.predict(X_test_scaled)

print("Predictions generated successfully!")
print(y_pred_lr[:10])

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred_lr)
mse = mean_squared_error(y_test, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance")
print("-----------------------------")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2) # Our model explains approximately 82.55% of the variation in solar power output on the unseen test period.

results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [mae],
    "MSE": [mse],
    "RMSE": [rmse],
    "R2 Score": [r2]
})

results


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

# Generate Predictions
y_pred_rf = rf_model.predict(X_test)

print("Random Forest predictions generated successfully!")
print(y_pred_rf[:10])

# Evaluate Random Forest

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance")
print("--------------------------")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

# Lower MAE, MSE and RMSE are better. Higher R² is better.

results.loc[len(results)] = [
    "Random Forest",
    mae_rf,
    mse_rf,
    rmse_rf,
    r2_rf
]

results


In [ ]:
from sklearn.svm import LinearSVR

# Create the model
svr_model = LinearSVR(
    C=1.0,
    max_iter=10000,
    random_state=42
)

# Train the model
svr_model.fit(X_train_scaled, y_train)

print("SVR model trained successfully!")

# Generate SVR Predictions 

y_pred_svr = svr_model.predict(X_test_scaled)

print("SVR predictions generated successfully!")
print(y_pred_svr[:10])

# Evaluate SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("SVR Performance")
print("----------------")
print("MAE:", mae_svr)
print("MSE:", mse_svr)
print("RMSE:", rmse_svr)
print("R2 Score:", r2_svr)


# Performance Deviation Analysis


In [ ]:
# Create a DataFrame for residual analysis

residual_df = pd.DataFrame({
    "Time": time_test,
    "Actual_Power": y_test,
    "Predicted_Power": y_pred_rf
})

# Calculate residuals
residual_df["Residual"] = (
    residual_df["Actual_Power"] -
    residual_df["Predicted_Power"]
)

# Calculate absolute residuals
residual_df["Absolute_Residual"] = residual_df["Residual"].abs()

print("Residuals calculated successfully!")

residual_df.head(10)

plt.figure(figsize=(15, 6))

plt.plot(
    residual_df["Time"],
    residual_df["Actual_Power"],
    label="Actual Power",
    alpha=0.7
)

plt.plot(
    residual_df["Time"],
    residual_df["Predicted_Power"],
    label="Predicted Power",
    alpha=0.7
)

plt.title("Actual vs Predicted Solar Power Generation")
plt.xlabel("Time")
plt.ylabel("Power (W)")

plt.legend()
plt.grid(True)

plt.show()

plt.figure(figsize=(10, 6))

plt.hist(residual_df["Residual"], bins=50, edgecolor="black")

plt.title("Distribution of Prediction Residuals")
plt.xlabel("Residual (Actual Power - Predicted Power)")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

residual_df["Residual"].describe()

# identify high deviation periods 

# Calculate the 95th percentile threshold
threshold = residual_df["Absolute_Residual"].quantile(0.95)

print("95th Percentile Deviation Threshold:", threshold, "W")
# This means any test observation with an absolute residual greater than 2770.72 W
# will be marked as a high-deviation period.

# Extract High-Deviation Observations
# Filter observations with high deviations
high_deviations = residual_df[
    residual_df["Absolute_Residual"] > threshold
]

print("Total High-Deviation Observations:", len(high_deviations))

print("\nFirst 10 High-Deviation Observations:")
high_deviations.head(10)

# Analyze Deviations by Hour of Day ⏰

# Extract hour from timestamps
high_deviations["Hour"] = high_deviations["Time"].dt.hour

# Count high deviations by hour
hourly_deviations = high_deviations["Hour"].value_counts().sort_index()

print(hourly_deviations)

# High deviations are concentrated during 8 AM–1 PM.
# The maximum occurs at 10 AM with 345 observations.
# Deviations become much less frequent after 2 PM.
# There are almost no high deviations at 4 PM and beyond in this result.

# Analyze deviations by Weather conditions

# Add weather type from the original test dataset
high_deviations["Weather_Type"] = df.loc[
    high_deviations.index, "weather_type"
]

# Count high deviations by weather type
weather_deviations = high_deviations["Weather_Type"].value_counts().sort_index()

print(weather_deviations)

# Analyze deviations by Month

# Extract month from timestamp
high_deviations["Month"] = high_deviations["Time"].dt.month

# Count high deviations by month
monthly_deviations = high_deviations["Month"].value_counts().sort_index()

print(monthly_deviations)

# Analyze the Actual vs Predicted Difference by Weather Type 

# Add weather type to the full residual dataframe
residual_df["Weather_Type"] = df.loc[residual_df.index, "weather_type"]

# Calculate average absolute residual by weather type
weather_error_analysis = residual_df.groupby("Weather_Type")[
    "Absolute_Residual"
].mean()

print(weather_error_analysis)

# Analyze Deviations by GHI 

# Divide GHI into ranges
residual_df["GHI_Range"] = pd.cut(
    df.loc[residual_df.index, "GHI"],
    bins=[-1, 100, 300, 600, 900, float("inf")],
    labels=["0-100", "100-300", "300-600", "600-900", "900+"]
)

# Calculate average absolute residual for each GHI range
ghi_error_analysis = residual_df.groupby(
    "GHI_Range", observed=False
)["Absolute_Residual"].mean()

print(ghi_error_analysis)

df.loc[residual_df.index, "GHI"].describe()

df.loc[residual_df.index, "GHI"].max()

# Correct the GHI Analysis
residual_df["GHI_Range"] = pd.cut(
    df.loc[residual_df.index, "GHI"],
    bins=[-1, 5, 25, 50, 100, 300],
    labels=["0-5", "5-25", "25-50", "50-100", "100-300"]
)

ghi_error_analysis = residual_df.groupby(
    "GHI_Range", observed=False
)["Absolute_Residual"].mean()

print(ghi_error_analysis) 

# Higher irradiance levels correspond to higher solar generation, where changes in
# environmental conditions can lead to larger absolute prediction errors.

high_deviation_percentage = (
    len(high_deviations) / len(residual_df)
) * 100

print("High-Deviation Percentage:", high_deviation_percentage, "%")

# final deviation summary

# | Metric | Result |
# |---|---:|
# | Total test observations | 39,356 |
# | High-deviation observations | 1,968 |
# | Deviation threshold (95th percentile) | 2,770.72 W |
# | High-deviation percentage | 5.0005% |
# | Model R² score | 89.86% |
# | Random Forest MAE | 545.98 W |
# | Random Forest RMSE | 1,333.15 W |

# We used the 95th percentile of absolute residuals as the cutoff. 
#     Therefore, the top ~5% of observations are flagged for investigation.
#     The slight difference from exactly 5% is due to ties and the strict > threshold.


In [ ]:
from pathlib import Path
import joblib

# Resolve the Model folder whether the notebook is opened from the project root or Model folder
current_dir = Path.cwd()
if current_dir.name == "Model":
    MODEL_DIR = current_dir
elif (current_dir / "Model").is_dir():
    MODEL_DIR = current_dir / "Model"
else:
    raise FileNotFoundError("Model folder not found. Open this notebook from Solar Power or Solar Power/Model.")

# Save the trained Random Forest model inside Model
model_path = MODEL_DIR / "solar_power_model.joblib"
joblib.dump(rf_model, model_path)
print("Model saved successfully!")
print("Saved at:", model_path.resolve())

# Load the saved model
loaded_model = joblib.load(model_path)
print("Model loaded successfully!")
print(type(loaded_model))

# Save the exact feature names and order used by Random Forest
feature_path = MODEL_DIR / "feature_names.joblib"
joblib.dump(X_train.columns.tolist(), feature_path)
print("Feature names saved successfully!")
print("Saved at:", feature_path.resolve())
